In [2]:
from src.db import init_mongo
import os

# === Initialize MongoDB ===
uri = os.getenv("MONGODB_URI")
mongo_client = init_mongo()
db = mongo_client["KB_PROPERTY_LAW"]
documents_col = db["documents"]
sections_col = db["legal_sections"]
process_sections_col = db["processed_legal_sections"]
relations_col = db["relations"]
concepts_col = db["concepts"]
triplets_col = db["triplets"]

You successfully connected to MongoDB!


In [2]:
from src.triplet_extraction.pos_taging import init_vncorenlp
import phonlp

vncorenlp_client = init_vncorenlp(r"/triplet_extraction/nlp_models/VnCoreNLP-1.2")
phoNLP_model = phonlp.load(save_dir=r"/triplet_extraction/nlp_models/phonlp")

Loading model from: E:\Github\LawAssistant\triplet_extraction\nlp_models\phonlp/phonlp.pt


In [3]:
# === Function to check if a word/phrase matches name or synonyms ===
def find_matches(word, items):
    """
    Find items where word matches the name or any synonym
    Returns list of matching items
    """
    matches = []
    for item in items:
        # Check exact match with name
        if item.get('name', '').lower() == word.lower():
            matches.append(item)
            continue

        # Check match in synonyms array
        synonyms = item.get('synonyms', [])
        if isinstance(synonyms, list):
            for syn in synonyms:
                if isinstance(syn, str) and syn.lower() == word.lower():
                    matches.append(item)
                    break

    return matches

# === Function to check multi-word phrases ===
def find_phrase_matches(segmented_text, start_idx, max_length, items):
    """
    Check for matches starting from start_idx for phrases up to max_length words
    Returns (matched_items, phrase_length) or (None, 0) if no match
    """
    best_match = None
    best_length = 0

    # Try phrases from longest to shortest
    for length in range(min(max_length, len(segmented_text) - start_idx), 0, -1):
        phrase = " ".join(segmented_text[start_idx:start_idx + length])
        matches = find_matches(phrase, items)

        if matches:
            if length > best_length:
                best_match = matches
                best_length = length

    return best_match, best_length

In [7]:
import os
from src.triplet_extraction.src import init_mongo
from src.triplet_extraction.src import clean_text, parsing_result

# === Initialize MongoDB ===
uri = os.getenv("MONGODB_URI")
mongo_client = init_mongo()
db = mongo_client["KB_PROPERTY_LAW"]
documents_col = db["documents"]
sections_col = db["legal_sections"]
process_sections_col = db["processed_legal_sections"]
relations_col = db["relations"]
concepts_col = db["concepts"]
triplets_col = db["triplets"]

# === Process text ===
text = "Phải xác nhận tài sản trên đất mới được bán đất có đúng không?"
question = clean_text(text)
segmented_text = vncorenlp_client.word_segment(question)[0]
print("Segmented text:", segmented_text)

annotation = phoNLP_model.annotate(text=segmented_text)
df = parsing_result(annotation)
print(df.to_string(index=False))

segmented_text = segmented_text.split(" ")

# === Fetch all concepts and relations ===
all_concepts = list(concepts_col.find({}))
all_relations = list(relations_col.find({}))

# === Function to check if a word/phrase matches name or synonyms ===
def find_matches(word, items):
    """
    Find items where word matches the name or any synonym
    Returns list of matching items
    """
    matches = []
    for item in items:
        # Check exact match with name
        if item.get('name', '').lower() == word.lower():
            matches.append(item)
            continue

        # Check match in synonyms array
        synonyms = item.get('synonyms', [])
        if isinstance(synonyms, list):
            for syn in synonyms:
                if isinstance(syn, str) and syn.lower() == word.lower():
                    matches.append(item)
                    break

    return matches

# === Function to check multi-word phrases ===
def find_phrase_matches(segmented_text, start_idx, max_length, items):
    """
    Check for matches starting from start_idx for phrases up to max_length words
    Returns (matched_items, phrase_length) or (None, 0) if no match
    """
    best_match = None
    best_length = 0

    # Try phrases from longest to shortest
    for length in range(min(max_length, len(segmented_text) - start_idx), 0, -1):
        phrase = " ".join(segmented_text[start_idx:start_idx + length])
        matches = find_matches(phrase, items)

        if matches:
            if length > best_length:
                best_match = matches
                best_length = length

    return best_match, best_length

# === Main matching logic ===
matched_concepts = []
matched_relations = []
i = 0
max_phrase_length = 5  # Adjust based on expected phrase lengths

while i < len(segmented_text):
    word = segmented_text[i]

    # Try to match phrases (multi-word)
    concept_matches, concept_length = find_phrase_matches(
        segmented_text, i, max_phrase_length, all_concepts
    )
    relation_matches, relation_length = find_phrase_matches(
        segmented_text, i, max_phrase_length, all_relations
    )

    # Prioritize longer matches
    if concept_length > 0 or relation_length > 0:
        if concept_length >= relation_length:
            for match in concept_matches:
                matched_concepts.append({
                    'position': i,
                    'matched_text': " ".join(segmented_text[i:i + concept_length]),
                    'data': match
                })
            i += concept_length
        else:
            for match in relation_matches:
                matched_relations.append({
                    'position': i,
                    'matched_text': " ".join(segmented_text[i:i + relation_length]),
                    'data': match
                })
            i += relation_length
    else:
        i += 1

# === Display results ===
print("\n=== Matched Concepts (in order of appearance) ===")
for match in matched_concepts:
    print(f"Position {match['position']}: '{match['matched_text']}'")
    print(f"  Name: {match['data'].get('name')}")
    print(f"  ID: {match['data'].get('_id')}")
    print(f"  Synonyms: {match['data'].get('synonyms', [])}")
    print()

print("\n=== Matched Relations (in order of appearance) ===")
for match in matched_relations:
    print(f"Position {match['position']}: '{match['matched_text']}'")
    print(f"  Name: {match['data'].get('name')}")
    print(f"  ID: {match['data'].get('_id')}")
    print(f"  Synonyms: {match['data'].get('synonyms', [])}")
    print()

# === Create combined ordered array ===
all_matches = []
for match in matched_concepts:
    all_matches.append({
        'type': 'concept',
        'position': match['position'],
        'matched_text': match['matched_text'],
        'data': match['data']
    })

for match in matched_relations:
    all_matches.append({
        'type': 'relation',
        'position': match['position'],
        'matched_text': match['matched_text'],
        'data': match['data']
    })

# Sort by position to maintain appearance order
all_matches.sort(key=lambda x: x['position'])

print("\n=== All Matches (ordered by appearance) ===")
for match in all_matches:
    print(f"[{match['type'].upper()}] Position {match['position']}: '{match['matched_text']}' -> {match['data'].get('name')}")

You successfully connected to MongoDB!
Segmented text: phải xác_nhận tài_sản trên đất mới được bán đất có đúng không


100%|██████████| 1/1 [00:00<00:00, 12.59it/s]

 id     word pos head deprel
  1     phải   V    0   root
  2 xác_nhận   V    1   vmod
  3  tài_sản   N    2    dob
  4     trên   E    3    loc
  5      đất   N    4    pob
  6      mới   R    8    adv
  7     được   V    8    adv
  8      bán   V    1   vmod
  9      đất   N    8    dob
 10       có   V    8   vmod
 11     đúng   A   10   amod
 12    không   R    1  punct



=== Matched Concepts (in order of appearance) ===
Position 0: 'phải'
  Name: phải
  ID: 696204c797fdc5f42bd6b74f
  Synonyms: []

Position 3: 'trên'
  Name: trên
  ID: 6962024997fdc5f42bd6ac8a
  Synonyms: []

Position 4: 'đất'
  Name: đất
  ID: 6962006b97fdc5f42bd6a3b1
  Synonyms: []

Position 5: 'mới'
  Name: mới
  ID: 69620f8097fdc5f42bd6e4d3
  Synonyms: []

Position 6: 'được'
  Name: được
  ID: 6962037997fdc5f42bd6b1c0
  Synonyms: []

Position 8: 'đất có'
  Name: đất có
  ID: 6962011c97fdc5f42bd6a70e
  Synonyms: []

Position 10: 'đúng'
  Name: đúng
  ID: 696203ac97fdc5f42bd6b295
  Synonyms: []

Position 11: 'không'
  Name: không
  ID: 6962023497fdc5f42bd6ac30
  Synonyms: []


=== Matched Relations (in order of appearance) ===
Position 7: 'bán'
  Name: bán
  ID: 696200ba97fdc5f42bd6a59f
  Synonyms: []


=== All Matches (ordered by appearance) ===
[CONCEPT] Position 0: 'phải' -> phải
[CONCEPT] Position 3: 'trên' -> trên
[CONCEPT] Position 4: 'đất' -> đất
[CONCEPT] Position 5: 'mới' ->

In [ ]:
text = "Phải xác nhận tài sản trên đất mới được bán đất có đúng không?"

In [18]:
import os
from collections import defaultdict
from src.triplet_extraction.src import init_mongo
from src.triplet_extraction.src import clean_text

# === Initialize MongoDB ===
uri = os.getenv("MONGODB_URI")
mongo_client = init_mongo()
db = mongo_client["KB_PROPERTY_LAW"]
documents_col = db["documents"]
sections_col = db["legal_sections"]
process_sections_col = db["processed_legal_sections"]
relations_col = db["relations"]
concepts_col = db["concepts"]
triplets_col = db["triplets_new"]

# === Process text ===
text = "Điều kiện chuyển nhượng quyền sử dụng đất là gì?"
question = clean_text(text)
segmented_text = vncorenlp_client.word_segment(question)[0]
print("Segmented text:", segmented_text)

segmented_text = segmented_text.split(" ")

# === Fetch all concepts and relations ===
all_concepts = list(concepts_col.find({}))
all_relations = list(relations_col.find({}))

# === Function to check if a word/phrase matches name or synonyms ===
def find_matches(word, items):
    """
    Find items where word matches the name or any synonym
    Returns list of matching items
    """
    matches = []
    for item in items:
        # Check exact match with name
        if item.get('name', '').lower() == word.lower():
            matches.append(item)
            continue

        # Check match in synonyms array
        synonyms = item.get('synonyms', []) or item.get('synonym', [])
        if isinstance(synonyms, list):
            for syn in synonyms:
                if isinstance(syn, str) and syn.lower() == word.lower():
                    matches.append(item)
                    break

    return matches

# === Function to check multi-word phrases ===
def find_phrase_matches(segmented_text, start_idx, max_length, items):
    """
    Check for matches starting from start_idx for phrases up to max_length words
    Returns (matched_items, phrase_length) or (None, 0) if no match
    """
    best_match = None
    best_length = 0

    # Try phrases from longest to shortest
    for length in range(min(max_length, len(segmented_text) - start_idx), 0, -1):
        phrase = " ".join(segmented_text[start_idx:start_idx + length])
        matches = find_matches(phrase, items)

        if matches:
            if length > best_length:
                best_match = matches
                best_length = length

    return best_match, best_length

# === Main matching logic ===
matched_concepts = []
matched_relations = []
i = 0
max_phrase_length = 5  # Adjust based on expected phrase lengths

while i < len(segmented_text):
    word = segmented_text[i]

    # Try to match phrases (multi-word)
    concept_matches, concept_length = find_phrase_matches(
        segmented_text, i, max_phrase_length, all_concepts
    )
    relation_matches, relation_length = find_phrase_matches(
        segmented_text, i, max_phrase_length, all_relations
    )

    # Prioritize longer matches
    if concept_length > 0 or relation_length > 0:
        if concept_length >= relation_length:
            for match in concept_matches:
                matched_concepts.append({
                    'position': i,
                    'matched_text': " ".join(segmented_text[i:i + concept_length]),
                    'data': match
                })
            i += concept_length
        else:
            for match in relation_matches:
                matched_relations.append({
                    'position': i,
                    'matched_text': " ".join(segmented_text[i:i + relation_length]),
                    'data': match
                })
            i += relation_length
    else:
        i += 1

# === Display initial matches ===
print("\n=== Matched Concepts ===")
for match in matched_concepts:
    print(f"Position {match['position']}: '{match['matched_text']}' -> {match['data'].get('name')}")

print("\n=== Matched Relations ===")
for match in matched_relations:
    print(f"Position {match['position']}: '{match['matched_text']}' -> {match['data'].get('name')}")


# ========================================
# === RETRIEVAL SYSTEM ===
# ========================================

def retrieve_relevant_sections(matched_concepts, matched_relations, top_k=10):
    """
    Retrieve relevant section_ids based on matched concepts and relations

    Strategy:
    1. Query triplets where matched concepts appear as subject or object
    2. Query triplets where matched relations appear
    3. Score sections based on triplet matches
    4. Return top-k sections with highest relevance
    """

    # Extract IDs from matches
    concept_ids = [match['data']['_id'] for match in matched_concepts]
    relation_ids = [match['data']['_id'] for match in matched_relations]

    print(f"\n=== Searching for {len(concept_ids)} concepts and {len(relation_ids)} relations ===")

    # Build query
    query_conditions = []

    # Match triplets with concepts as subject or object
    if concept_ids:
        query_conditions.append({
            '$or': [
                {'subject_id': {'$in': concept_ids}},
                {'object_id': {'$in': concept_ids}}
            ]
        })

    # Match triplets with relations
    if relation_ids:
        query_conditions.append({'relation_id': {'$in': relation_ids}})

    # Combine queries
    if not query_conditions:
        print("No concepts or relations matched. Cannot retrieve sections.")
        return []

    # Use OR to find triplets matching any condition
    query = {'$or': query_conditions} if len(query_conditions) > 1 else query_conditions[0]

    # Query triplets
    matched_triplets = list(triplets_col.find(query))
    print(f"Found {len(matched_triplets)} matching triplets")

    if not matched_triplets:
        print("No triplets found matching the concepts/relations.")
        return []

    # Score sections based on triplet matches
    section_scores = defaultdict(lambda: {
        'score': 0,
        'triplet_count': 0,
        'concept_matches': 0,
        'relation_matches': 0,
        'full_triplet_matches': 0,
        'so_hieu': set(),
        'matched_triplets': []
    })

    for triplet in matched_triplets:
        # Extract section information
        documents = triplet.get('documents', [])

        # Check match type
        has_subject = triplet.get('subject_id') in concept_ids
        has_object = triplet.get('object_id') in concept_ids
        has_relation = triplet.get('relation_id') in relation_ids

        # Calculate base score for this triplet
        triplet_score = 0
        if has_subject and has_object and has_relation:
            triplet_score = 10  # Full triplet match (all 3 components)
        elif (has_subject or has_object) and has_relation:
            triplet_score = 5   # Concept + relation match
        elif has_subject and has_object:
            triplet_score = 4   # Both concepts match
        elif has_subject or has_object:
            triplet_score = 2   # Single concept match
        elif has_relation:
            triplet_score = 1   # Relation only match

        # Update scores for each document/section
        for doc in documents:
            section_id = doc.get('section_id')
            so_hieu = doc.get('so_hieu')

            if section_id:
                section_scores[section_id]['score'] += triplet_score
                section_scores[section_id]['triplet_count'] += 1
                section_scores[section_id]['so_hieu'].add(so_hieu)

                if has_subject or has_object:
                    section_scores[section_id]['concept_matches'] += 1
                if has_relation:
                    section_scores[section_id]['relation_matches'] += 1
                if has_subject and has_object and has_relation:
                    section_scores[section_id]['full_triplet_matches'] += 1

                section_scores[section_id]['matched_triplets'].append({
                    'subject': triplet.get('subject_name'),
                    'relation': triplet.get('relation_name'),
                    'object': triplet.get('object_name'),
                    'score': triplet_score
                })

    # Convert to sorted list
    ranked_sections = []
    for section_id, data in section_scores.items():
        ranked_sections.append({
            'section_id': section_id,
            'score': data['score'],
            'triplet_count': data['triplet_count'],
            'concept_matches': data['concept_matches'],
            'relation_matches': data['relation_matches'],
            'full_triplet_matches': data['full_triplet_matches'],
            'so_hieu': list(data['so_hieu']),
            'matched_triplets': data['matched_triplets'][:5]  # Top 5 triplets for this section
        })

    # Sort by score (descending)
    ranked_sections.sort(key=lambda x: (x['score'], x['full_triplet_matches'], x['triplet_count']), reverse=True)

    return ranked_sections[:top_k]


# === Execute Retrieval ===
print("\n" + "="*80)
print("=== RETRIEVAL RESULTS ===")
print("="*80)

relevant_sections = retrieve_relevant_sections(matched_concepts, matched_relations, top_k=10)

if relevant_sections:
    print(f"\nFound {len(relevant_sections)} relevant sections:\n")

    for idx, section in enumerate(relevant_sections, 1):
        print(f"\n--- Rank {idx} ---")
        print(f"Section ID: {section['section_id']}")
        print(f"Score: {section['score']}")
        print(f"Legal Document(s): {', '.join(section['so_hieu'])}")
        print(f"Triplet Matches: {section['triplet_count']}")
        print(f"  - Concept Matches: {section['concept_matches']}")
        print(f"  - Relation Matches: {section['relation_matches']}")
        print(f"  - Full Triplet Matches: {section['full_triplet_matches']}")

        print(f"\nTop Matched Triplets:")
        for t_idx, triplet in enumerate(section['matched_triplets'], 1):
            print(f"  {t_idx}. [{triplet['subject']}] --{triplet['relation']}--> [{triplet['object']}] (score: {triplet['score']})")

        # Optionally fetch actual section content
        section_doc = sections_col.find_one({'section_id': section['section_id']})
        if section_doc:
            content = section_doc.get('content', '')
            preview = content[:200] + "..." if len(content) > 200 else content
            print(f"\nSection Preview: {preview}")
else:
    print("\nNo relevant sections found.")

# === Export section IDs for further use ===
print("\n" + "="*80)
print("=== SECTION IDs FOR RETRIEVAL ===")
print("="*80)
section_ids = [s['section_id'] for s in relevant_sections]
print(f"\nRetrieved {len(section_ids)} section IDs:")
for section_id in section_ids:
    print(f"  - {section_id}")

You successfully connected to MongoDB!
Segmented text: điều_kiện chuyển_nhượng quyền sử_dụng đất là gì

=== Matched Concepts ===
Position 2: 'quyền' -> quyền
Position 2: 'quyền' -> giấy phép
Position 4: 'đất là' -> đất là

=== Matched Relations ===

=== RETRIEVAL RESULTS ===

=== Searching for 3 concepts and 0 relations ===
Found 804 matching triplets

Found 10 relevant sections:


--- Rank 1 ---
Section ID: bb3197998b5c8961aa6a85285d6465d89d42d4eedd89043ba62e06e18f4c33aa
Score: 34
Legal Document(s): 102/2024/NĐ-CP
Triplet Matches: 17
  - Concept Matches: 17
  - Relation Matches: 0
  - Full Triplet Matches: 0

Top Matched Triplets:
  1. [quyền] --sử dụng--> [đất] (score: 2)
  2. [quyền] --quy định--> [đất] (score: 2)
  3. [quyền] --quy định--> [diện tích đất] (score: 2)
  4. [quyền] --quy định--> [văn phòng làm việc] (score: 2)
  5. [quyền] --quy định--> [cơ sở thương mại] (score: 2)

--- Rank 2 ---
Section ID: 39b330ee9443979fc099f86ec8750cda56b3924267bc07819f688b7f83c59e1e
Score: 24


In [19]:
for section_id in section_ids:
    section_doc = sections_col.find_one({'_id': section_id})
    if section_doc:
        content = section_doc.get('content', '')
        full_path = section_doc.get('full_path', 'N/A')
        print(f"\n=== Section ID: {section_id} ===")
        print(f"Full Path: {full_path}")
        print(content)


=== Section ID: bb3197998b5c8961aa6a85285d6465d89d42d4eedd89043ba62e06e18f4c33aa ===
Full Path: 102/2024/NĐ-CP_chương vii_mục 6_điều 92_khoản 3_điểm a
Diện tích đất thuộc quyền sử dụng chung của các chủ sở hữu căn hộ chung cư, văn phòng làm việc, cơ sở thương mại, dịch vụ trong nhà chung cư (sau đây gọi chung là căn hộ) bao gồm diện tích đất xây dựng khối nhà chung cư, làm sân, trồng hoa, cây xanh xung quanh nhà và đất xây dựng các công trình hạ tầng bên ngoài nhà chung cư nhưng để phục vụ trực tiếp cho nhà chung cư được chủ đầu tư bàn giao cho các chủ sở hữu căn hộ tự tổ chức quản lý, sử dụng theo dự án đầu tư. Chủ đầu tư có trách nhiệm xác định rõ vị trí, ranh giới, diện tích đất thuộc quyền sử dụng chung trong dự án đầu tư, quy hoạch xây dựng chi tiết và thiết kế mặt bằng công trình để trình cơ quan có thẩm quyền phê duyệt; trong các hợp đồng mua bán nhà ở và bản vẽ hoàn công công trình để làm thủ tục cấp Giấy chứng nhận cho bên mua;

=== Section ID: 39b330ee9443979fc099f86ec8750cd